# sLTN Running Example — Video Classification of Handwritten Digits

This notebook presents an end-to-end video-classification example over sequences of handwritten digits, implemented with the **sLTN** library. sLTN extends Real Logic with *structural dimensions*: named tensor axes, such as time, that are declared directly in the signature, appear in the syntax of formulas, and receive tensor semantics through an `Interpretation`. Users first declare symbols in a `Signature` and write formulas at the syntactic level; concrete tensor semantics are assigned later.

This notebook is organised as follows: signature → formulas/knowledge base → semantics (types and logic) → dataset → models → training → evaluation.

For further details, see the accompanying paper: **[arXiv: TBD](https://arxiv.org/abs/2608.11136)**.

**Running example.** A video is a sequence of frames. Some videos show a digit genuinely *appearing*: the digit is progressively revealed and then remains visible. Other videos show a digit appear and then disappear again. We declare:

- sorts `Image` (frames) and `Digit` (labels);
- a structural dimension `T` indexing time steps, with structural variable `t` ranging over it;
- a variable `x` representing a video and carrying dimension `T`;
- a variable `y` representing a per-video label, with no `T` dimension;
- a constant `unknown` of sort `Digit`, used both for incomplete frames and for videos that are not genuine appearing-digit videos;
- a function `digit`, a **frame-local** classifier with no structural dimensions declared: its type provides no basis for recurrence;
- a Boolean predicate `Complete`, indicating whether the digit is fully drawn in a given frame;
- a Boolean predicate `appear`, indicating whether the *whole video* is a genuine appearing-digit-`y` video; this predicate consumes `T` and produces no structural dimension, so it structurally requires a whole-sequence model;
- an infix equality predicate `=d` between digits;
- a structural relation `next` between consecutive time points.

We state five formulas that together form the knowledge base $\mathcal K$. The first 2 formulas use ground truth directly:

$$
\delta := \forall (x, y): \mathrm{appear}(x, y)
$$

$$
\phi := \forall (x, y): \mathrm{appear}(x, y) \wedge \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{digit}(x[t]) =_d y\bigr)
$$

$$
\psi := \forall x: \mathrm{is\_appear}(x) \rightarrow \forall t, t_1 \mid \mathrm{next}(t, t_1): \bigl(\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{Complete}(x[t_1])[t_1]\bigr)
$$

$$
\chi := \forall x: \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\neg\,\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{digit}(x[t]) =_d \mathrm{unknown}\bigr)
$$

$$
\beta := \forall x: \mathrm{is\_appear}(x) \rightarrow \bigl(\neg\,\mathrm{Complete}(x[t])[t{=}0]\ \wedge\ \mathrm{Complete}(x[t])[t{=}T{-}1]\bigr)
$$

where $\mathrm{is\_appear}(x) := \neg\,\mathrm{appear}(x, \mathrm{unknown})$.

The clause $\delta$ is the **only directly supervised clause**: it pushes `appear(x, y)` towards the ground-truth video-level label. All other clauses relate `digit` and `Complete` to `appear`'s own prediction. Thus, these frame-level symbols are learned indirectly, as a consequence of satisfying a knowledge base whose only supervised anchor is a coarse video-level judgement.

The non-appearing, or “appear-then-disappear”, videos make the task non-trivial. Without them, the statement “this is an appearing digit” would always be true, and the gates in $\psi$, $\chi$, and $\beta$ would have little effect. The clause $\phi$ binds `digit` at *every* `Complete` frame, not just one, so that `Complete` cannot trivially fire too early or everywhere: doing so would force `digit` to classify ambiguous, partially revealed frames correctly.

We interpret `appear` with a **recurrent model**: a per-frame CNN encoder followed by a GRU, read out from its final hidden state. This illustrates the role of a structurally necessary whole-sequence symbol. In contrast, `digit` and `Complete` are interpreted with per-frame CNN classifiers. In particular, no recurrent grounding is well typed for `digit`, since its signature declares no structural dimensions. We then build a synthetic dataset of appearing and appear-then-disappear MNIST videos, train all three models jointly against $\mathcal K$, and finally probe whether `digit` and `Complete` learned to track the real digit label and the true completeness trace, even though neither was directly supervised.

In [ ]:
%load_ext autoreload
%autoreload 2

## §1  Configuration

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
NUM_SAMPLES_TRAIN = 3000
NUM_SAMPLES_VAL   = 600
NUM_SAMPLES_TEST  = 600
BATCH_SIZE        = 32
TIME_STEPS        = 24     # length of a video
COMPLETE_FRAC     = 0.85   # a digit counts as "complete" once this much of it is revealed
APPEAR_FRAC       = 0.6    # fraction of videos that are genuine "appear" (monotone reveal);
                           # the rest "appear-then-disappear" (digit fades back out)
NOISE_STD         = 0.03
RANDOM_SEED       = 2

# ── Training ───────────────────────────────────────────────────────────────
NUM_EPOCHS   = 15
LR           = 1e-3
WEIGHT_DECAY = 1e-5
MAX_NORM     = 1.0
PATIENCE     = 8

# ── Model ──────────────────────────────────────────────────────────────────
FRAME_FEATURE_DIM = 64
RNN_HIDDEN_DIM    = 64
N_DIGITS          = 10
N_CLASSES         = N_DIGITS + 1   # classes 0-9 = real digits; class 10 = "unknown/incomplete"

## §2  Imports

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

import sltn
from sltn import Signature, Interpretation, Logic, KB, kb_describe, kb_backward
import sltn.interpretation.operators as ops


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## §3  Signature

An sLTN signature is a tuple $\Sigma=(\mathcal S,\mathcal D,\mathcal C,\mathcal X,\mathcal F,\mathcal P,\mathcal{SV},\mathcal{SR})$ consisting of sorts, structural dimensions, constants, variables, functions, predicates, structural variables, and structural relations.

This running example highlights symbols that interact differently with the same structural axis `T`:

| Signature construct | `Signature` method | sLTN code |
|---|---|---|
| signature $\Sigma$ | `Signature(name)` | `sig = Signature("video")` |
| sort `Image` | `sig.sort(name)` | `sig.sort("Image")` |
| sort `Digit` | `sig.sort(name)` | `sig.sort("Digit")` |
| dimension `T` | `sig.dimension(name)` | `sig.dimension("T")` |
| variable $x$ | `sig.variable(name, sort, dims=…)` | `sig.variable("x", "Image", dims=["T"])` |
| variable $y$ | `sig.variable(name, sort)` | `sig.variable("y", "Digit")` |
| structural variable $t$ | `sig.structural_variable(name, dim)` | `sig.structural_variable("t", "T")` |
| function `digit` | `sig.function(name, in_sorts, out_sort)` | `sig.function("digit", ["Image"], "Digit")` |
| predicate `Complete` | `sig.predicate(name, in_sorts, input_dims=…, output_dims=…)` | `sig.predicate("Complete")` |
| predicate `appear` | `sig.predicate(name, in_sorts, input_dims=…, output_dims=…)` | `sig.predicate("appear", ["Image", "Digit"], input_dims=["T"], output_dims=[])` |
| predicate `=d` | `sig.predicate(name, in_sorts, infix=True)` | `sig.predicate("=d", ["Digit", "Digit"], infix=True)` |
| constant `unknown` | `sig.constant(name, sort)` | `sig.constant("unknown", "Digit")` |
| structural relation `next` | `sig.structural_relation(name, dims)` | `sig.structural_relation("next", ["T", "T"])` |

A few points are worth highlighting:

- `digit` declares **no structural dimensions**. It is a plain `Image -> Digit` function: applying it to a frame produces a prediction with no notion of “before” or “after”. A frame-local CNN is therefore not merely a modelling option; it is the natural well-typed grounding admitted by the symbol's signature.

- `Complete` declares **no structural dimensions**. It is a plain `Image -> Bool` map.

- `appear` consumes `T` and produces no structural dimension (`output_dims=[]`). A time-independent per-frame model cannot produce a tensor of the required shape for `appear`. It is therefore the only symbol in this signature for which an axis-aware, whole-sequence grounding, such as a GRU, is structurally necessary.

- `y` is a **per-video** label with no `T` dimension. It denotes either the drawn digit, if the video is a genuine appearing-digit video, or `unknown`, if the video is not genuine appearing. It is the target of `appear(x, y)`, the only directly supervised symbol in this notebook (§4).

- `unknown` is a **constant** of sort `Digit`, grounded as a one-hot vector at index 10. It plays two roles: `chi` allows `digit` to abstain on incomplete frames, and `appear(x, unknown)` represents a video that is not a genuine appearing-digit video. The latter is used in the macro `is_appear` (§4).

In [ ]:
sig = Signature("video")

# Sorts
sig.sort("Image")
sig.sort("Digit")

# Structural dimension: time
sig.dimension("T")

# Variables: x = video (frame sequence, carries T); y = a per-video label (no T at all).
sig.variable("x", "Image", dims=["T"])
sig.variable("y", "Digit")

sig.structural_variable("t", "T")

# Function: digit classifier. No input_dims/output_dims at all — genuinely frame-local,
# no structural necessity for recurrence (contrast with `appear` below).
sig.function("digit", ["Image"], "Digit")

# Predicate: is the digit fully drawn at this frame? Still consumes-and-reproduces T.
sig.predicate("Complete", ["Image"], input_dims=[], output_dims=[])

# Predicate: is this whole video a genuine "appearing digit y" video? Consumes T,
# produces nothing back (output_dims=[]) — a whole-sequence judgement, structurally
# requiring a model that sees every frame (e.g. a GRU) to even be well-typed.
sig.predicate("appear", ["Image", "Digit"], input_dims=["T"], output_dims=[])

# Infix equality between digits
sig.predicate("=d", ["Digit", "Digit"], infix=True)

# Constant: the "unknown/not-appearing" pseudo-class — a one-hot at position N_DIGITS.
# Used both by chi (digit abstains on incomplete frames) and by appear (a video that is
# not a genuine appearing-digit video is "appear(x, unknown)").
sig.constant("unknown", "Digit")

# Structural relation: adjacency between consecutive time points
sig.structural_relation("next", ["T", "T"])

print(sig)

## §4  Formulas and Knowledge Base — weak supervision through a single coarse label

Only **one** formula uses ground truth directly: `delta`. Every other clause relates `digit` and `Complete` to `appear`, and never to the dataset fields `label` or `complete_gt`. Those fields are retained only for post-training probes, allowing us to check whether the two frame-level symbols learned their intended roles.

We first define a shorthand macro, which is a formula alias rather than a new signature symbol:

$$
\mathrm{is\_appear}(x) := \neg\,\mathrm{appear}(x, \mathrm{unknown})
$$

This reads as: “$x$ is genuinely a video of some appearing digit.” It is true when `appear` does not assign the video to the `unknown` / “not appearing” class.

$$
\delta := \forall (x, y): \mathrm{appear}(x, y)
$$

$$
\phi := \forall (x, y): \mathrm{appear}(x, y) \wedge \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{digit}(x[t]) =_d y\bigr)
$$

$$
\psi := \forall x: \mathrm{is\_appear}(x) \rightarrow \forall t, t_1 \mid \mathrm{next}(t, t_1): \bigl(\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{Complete}(x[t_1])[t_1]\bigr)
$$

$$
\chi := \forall x: \mathrm{is\_appear}(x) \rightarrow \forall t: \bigl(\neg\,\mathrm{Complete}(x[t])[t] \rightarrow \mathrm{digit}(x[t]) =_d \mathrm{unknown}\bigr)
$$

$$
\beta := \forall x: \mathrm{is\_appear}(x) \rightarrow \bigl(\neg\,\mathrm{Complete}(x[t])[t{=}0]\ \wedge\ \mathrm{Complete}(x[t])[t{=}T{-}1]\bigr)
$$

The five clauses play the following roles:

- $\delta$ — **direct supervision**: `appear(x, y)` should be true, where `y` is the video's ground-truth digit-or-unknown label. This is the only coarse supervised signal used for training the knowledge base.

- $\phi$ — **label matching at every `Complete` frame, gated by `appear` and `is_appear`**: if `x` is a genuine appearing-digit-`y` video, then every frame that `Complete` marks as fully revealed must be classified by `digit` as `y`. Thus, `digit` and `Complete` are pushed towards the ground truth only indirectly, through `appear`'s prediction.

  Requiring the condition at all `Complete` frames, rather than at a single existentially chosen frame, discourages degenerate solutions. If `Complete` fires too early or on every frame, `digit` would be forced to classify ambiguous, partially revealed frames correctly. The `is_appear` gate is also essential: without it, fully visible frames in disappear videos could force `digit(x[t]) =d unknown`, while visually identical frames in appear videos would force `digit(x[t]) =d label`, creating an inconsistency that a frame-local classifier cannot resolve.

- $\psi$ — **monotone completeness, gated by `is_appear`**: for genuine appearing-digit videos, once `Complete` becomes true, it should remain true. This is not asserted for non-appearing videos, making appear-then-disappear videos legitimate counterexamples rather than violations of the knowledge base.

- $\chi$ — **abstention on incomplete frames, gated by `is_appear`**: on appearing-digit videos, incomplete frames should be classified as `unknown`.

- $\beta$ — **boundary condition, gated by `is_appear`**: appearing-digit videos should start incomplete and end complete.

The knowledge base is

$$
\mathcal K = \{\delta,\ \phi,\ \forall x:\psi(x),\ \forall x:\chi(x),\ \forall x:\beta(x)\}.
$$

In code, we use `sig.define` to register each formula as a named macro, including `is_appear`, and then parse the knowledge base using the `{ … }` brace syntax.

In [ ]:
# Shorthand macro: "x is a genuine appearing-digit video" — never touches ground truth,
# only appear's own output.
sig.define("is_appear", ["x"], "~appear(x[t], unknown)", input_dims=["T"], output_dims=[])

sig.define("delta", [], "∀ (x, y): appear(x[t], y)")

sig.define("phi", [], "∀ (x, y): ((appear(x[t], y) & is_appear(x[t])) -> (∀ t: (Complete(x[t])[t] -> digit(x[t]) =d y)))")

# psi/chi/beta: gated by is_appear(x) — asserted only for genuine appearing-digit videos,
# so "appear-then-disappear" videos are legitimate counterexamples, not violations.
sig.define("psi", ["x"], "(is_appear(x[t]) -> (∀ t, t1 | next(t, t1): (Complete(x[t])[t] -> Complete(x[t1])[t1])))")
sig.define("chi", ["x"], "(is_appear(x[t]) -> (∀ t: ((¬Complete(x[t])[t] -> (digit(x[t]) =d unknown)))))", input_dims=["T"], output_dims=[])
beta_body = f"is_appear(x) -> (~Complete(x[t][t=0]) & Complete(x[t][t={TIME_STEPS-1}]))"
sig.define("beta", ["x"], beta_body)

delta = sig.parse("delta()")
phi   = sig.parse("phi()")
psi   = sig.parse("∀ x: psi(x)")
chi   = sig.parse("∀ x: chi(x)")
beta  = sig.parse("∀ x: beta(x)")

print("delta:", delta)
print("phi:  ", phi)
print("psi:  ", psi)
print("chi:  ", chi)
print("beta: ", beta)
print()
print("fv(delta):", delta.free_variables, " fsv(delta):", delta.free_structural_variables)
print("fv(phi)  :", phi.free_variables,   " fsv(phi)  :", phi.free_structural_variables)
print("fv(psi)  :", psi.free_variables,   " fsv(psi)  :", psi.free_structural_variables)

# K = {delta, phi, ∀x:psi, ∀x:chi, ∀x:beta}
phi_kb = sig.parse("{ delta(), phi(), ∀ x: psi(x), ∀ x: chi(x), ∀ x: beta(x) }")

print("\nKB:", phi_kb)
kb_dict = kb_describe(phi_kb)

## §5  Semantics — Types and Logic

**Types.** Each sort is assigned a concrete tensor type. We represent `Image` as a grayscale $1 \times 28 \times 28$ frame, and `Digit` as a length-**11** probability vector: indices 0–9 correspond to real digit classes, while index 10 is the `unknown` pseudo-class.

Ground-truth label vectors `y` place a `1` in the correct digit slot and `0` in the `unknown` slot. Consequently, the predicate $=_d$, implemented as a dot product, gives no reward for predicting `unknown` on labelled digit frames:

```python
interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
interp["Digit"] = sltn.Type("Digit", shape=(N_CLASSES,), axis_names=["digit_prob"])  # N_CLASSES = 11
```

**Logic.** Logical connectives, quantifiers, and knowledge-base aggregation are specified through a Logic object. We use the library defaults for ordinary connectives and quantifiers, and explicitly register structural aggregators for the axis `T`, as required for structural quantification. Structural aggregation has no library-wide default and must be set per dimension.

In [ ]:
STABLE, EPS = True, 1e-12

logic = Logic().with_defaults(stable=STABLE, eps=EPS)
logic["forall,t"] = ops.aggregator.AggregPMeanError(p=2, stable=STABLE, eps=EPS)
logic["exists,t"]  = ops.aggregator.AggregPMean(p=2, stable=STABLE, eps=EPS)
logic["->"] = ops.ImpliesLuk(stable=False)
logic["and"] = ops.AndLuk(stable=False)
logic["SatAgg"] = ops.aggregator.AggregPMeanError(p=2)

interp = Interpretation(sig, logic=logic).to(device)

interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
# N_CLASSES = N_DIGITS + 1: last position is the "unknown/incomplete" pseudo-class.
interp["Digit"] = sltn.Type("Digit", shape=(N_CLASSES,), axis_names=["digit_prob"])

# Infix equality between two digit-probability vectors: agreement = inner product.
# Clamped to [0, 1] to guard against softmax/float32 rounding slightly outside bounds.
interp["=d"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)

# Ground the "unknown" constant: one-hot at position N_DIGITS (the last class).
_unknown_vec = torch.zeros(N_CLASSES, device=device)
_unknown_vec[N_DIGITS] = 1.0
interp["unknown"] = _unknown_vec

print(interp)

## §6  Dataset — appearing and appear-then-disappear videos

Each sample is a video $x$ of `TIME_STEPS` frames, generated from a single MNIST digit. A sample belongs to one of two types, chosen with probabilities `APPEAR_FRAC` and `1 - APPEAR_FRAC`:

- **Appear**: the video contains a single random reveal window $[t_\mathrm{start}, t_\mathrm{end}]$. The frame is blank before $t_\mathrm{start}$, the digit is linearly revealed between $t_\mathrm{start}$ and $t_\mathrm{end}$, and the fully revealed digit remains visible afterwards. The ground-truth completeness trace `complete_gt` (revealed fraction $\ge$ `COMPLETE_FRAC`) is monotone, consistent with $\psi$, and the per-video label $y$ is the one-hot vector of the drawn digit.

- **Disappear**: the video starts with the same initial reveal, followed by a random number of additional hide/reveal cycles at random positions and durations. The digit may fade out, fade back in, fade out again, and end in an arbitrary visible or hidden state. The trace `complete_gt` is generally **non-monotone**, making it a genuine counterexample to $\psi$. The video-level label is $y = \mathrm{onehot}(\mathrm{unknown})$, meaning that the video is not a genuine appearing-digit video, even though a real digit was drawn and may be visible in some frames.

Only $y$ is used by the knowledge base, through `delta` and `appear`. The fields `label`, which stores the digit that was actually drawn, and `complete_gt`, which stores the true completeness trace, are used only after training to probe whether `digit` and `Complete` learned their intended behaviour without direct supervision.

In [ ]:
class AppearVideoDataset(Dataset):
    """Wraps an MNIST split into two kinds of videos.

    - "Appear" (probability `appear_frac`): a single reveal window [t_start, t_end] —
      blank before t_start, linearly revealed from t_start to t_end, fully revealed
      forever after. `complete_gt` is monotone. `y = onehot(label)`.
    - "Disappear" (probability 1 - appear_frac): the same initial reveal window,
      followed by a *random number* (1-3) of further hide/reveal cycles — the digit
      fades out, may fade back in, may fade out again, etc. — at random positions,
      durations, and with a random final state (it may end the video visible or
      hidden). `complete_gt` is generally non-monotone. `y = onehot(unknown)` — this
      is not a genuine "appearing digit" video, even though a real digit was drawn
      in it and may well be visible in the final frame.

    Only `y` (and, implicitly, `is_appear`) feed any formula. `label`/`complete_gt`
    are recorded purely for post-training probing of `digit`/`Complete`.
    """

    def __init__(self, mnist_split, num_samples, time_steps, complete_frac,
                 appear_frac=1.0, noise_std=0.0, seed=0):
        self.mnist_split   = mnist_split
        self.time_steps    = time_steps
        self.complete_frac = complete_frac
        self.appear_frac   = appear_frac
        self.noise_std     = noise_std

        rng = random.Random(seed)
        n_available = len(mnist_split)
        self.indices    = [rng.randrange(n_available) for _ in range(num_samples)]
        self.is_appear  = [rng.random() < appear_frac for _ in range(num_samples)]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.mnist_split[self.indices[idx]]
        H = image.shape[-2]
        is_appear = self.is_appear[idx]
        T = self.time_steps
        quarter = max(1, T // 4)

        rng_local = random.Random(hash((idx, "reveal")))
        t_start = rng_local.randrange(0, quarter + 1)
        t_end   = min(t_start + rng_local.randrange(0, quarter + 1), T // 2 - 1)

        reveal_frac = torch.zeros(T)
        if t_end > t_start:
            reveal_frac[t_start:t_end + 1] = torch.linspace(0.0, 1.0, steps=t_end - t_start + 1)
        else:
            reveal_frac[t_start] = 1.0
        reveal_frac[t_end + 1:] = 1.0    # fully revealed after t_end

        if not is_appear:
            # A random number of further hide/reveal cycles after the initial reveal:
            # fade out, hold, maybe fade back in, hold, fade out again, … — random
            # count, positions, and durations, ending in an arbitrary (visible or
            # hidden) final state. None of this is ever supervised directly; it just
            # produces varied, generally non-monotone `complete_gt` traces that serve
            # as counterexamples to psi (which only applies to genuine `is_appear`
            # videos).
            n_cycles = rng_local.randrange(1, 4)   # 1..3 fade transitions
            cursor   = t_end + 1
            visible  = True                        # state coming out of the initial reveal
            for _ in range(n_cycles):
                remaining = T - 1 - cursor
                if remaining <= 0:
                    break
                start_val, end_val = (1.0, 0.0) if visible else (0.0, 1.0)

                ramp_dur = rng_local.randrange(1, max(2, remaining // 2 + 1))
                ramp_end = min(cursor + ramp_dur, T - 1)
                if ramp_end > cursor:
                    reveal_frac[cursor:ramp_end + 1] = torch.linspace(start_val, end_val, steps=ramp_end - cursor + 1)
                else:
                    reveal_frac[cursor] = end_val

                hold_remaining = T - 1 - ramp_end
                hold_dur = rng_local.randrange(0, hold_remaining // 2 + 1) if hold_remaining > 0 else 0
                hold_end = min(ramp_end + hold_dur, T - 1)
                reveal_frac[ramp_end + 1:hold_end + 1] = end_val

                cursor, visible = hold_end + 1, not visible
                if cursor >= T - 1:
                    break

        frames = torch.zeros(T, *image.shape)
        for t, frac in enumerate(reveal_frac):
            n_rows = int(round(frac.item() * H))
            frames[t, :, :n_rows, :] = image[:, :n_rows, :]
        if self.noise_std > 0:
            frames = (frames + self.noise_std * torch.randn_like(frames)).clamp(0.0, 1.0)

        # y is a SINGLE per-video one-hot over N_CLASSES (no T!) — digit-index if this is
        # a genuine appearing-digit video, unknown-index otherwise. This is the only
        # ground truth any formula ever sees (via delta/phi's appear(x, y)).
        y_onehot = torch.zeros(N_CLASSES)
        y_onehot[label if is_appear else N_DIGITS] = 1.0

        complete_gt = (reveal_frac >= self.complete_frac).float().unsqueeze(-1)

        return {
            "x": frames, "y": y_onehot, "complete_gt": complete_gt,
            "label": label, "is_appear": float(is_appear),
        }


frame_size = (28, 28)
mnist_train_full = datasets.MNIST("../datasets/", train=True, download=False,
    transform=transforms.Compose([transforms.Resize(frame_size), transforms.ToTensor()]),
)
train_split_raw, val_split_raw = random_split(
    mnist_train_full, [50000, 10000],
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
mnist_test = datasets.MNIST("../datasets/", train=False, download=False,
    transform=transforms.Compose([transforms.Resize(frame_size), transforms.ToTensor()]),
)

train_ds = AppearVideoDataset(train_split_raw, NUM_SAMPLES_TRAIN, TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED)
val_ds   = AppearVideoDataset(val_split_raw,   NUM_SAMPLES_VAL,   TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED + 1)
test_ds  = AppearVideoDataset(mnist_test,       NUM_SAMPLES_TEST,  TIME_STEPS, COMPLETE_FRAC,
                               appear_frac=APPEAR_FRAC, noise_std=NOISE_STD, seed=RANDOM_SEED + 2)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

n_appear_train = sum(train_ds.is_appear)
print(f"Train: {len(train_ds)} ({n_appear_train} appear, {len(train_ds) - n_appear_train} disappear)")
print(f"Val:   {len(val_ds)}  |  Test: {len(test_ds)}")
print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

## §6.5  Dataset visualisation

Use the sliders to browse training samples and time steps. The left panel shows the current frame, and the right panel shows the ground-truth `Complete` trace over time. The trace is monotone for appear videos and typically rises and falls for disappear videos; the title indicates the video type.

In [ ]:
from ipywidgets import interact, IntSlider, fixed

def plot_sample_gt(sample_idx, time_step, dataset):
    sample = dataset[sample_idx]
    frames = sample["x"]                       # (T, 1, 28, 28)
    time_np = np.arange(TIME_STEPS)
    complete_np = sample["complete_gt"].squeeze(-1).numpy()

    fig, (ax_img, ax_tr) = plt.subplots(1, 2, figsize=(10, 4))

    kind = "appear" if sample["is_appear"] else "disappear"
    ax_img.imshow(frames[time_step, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax_img.set_title(f"frame t={time_step}  (label={sample['label']}, type={kind})")
    ax_img.axis("off")

    ax_tr.plot(time_np, complete_np, marker="o", color="tab:green")
    ax_tr.axvline(time_step, color="red", ls="--", alpha=0.7, label="current t")
    ax_tr.set_title(f"Ground-truth Complete(x)[t]  ({kind})")
    ax_tr.set_xlabel("time"); ax_tr.set_ylabel("truth value")
    ax_tr.set_ylim(-0.05, 1.1); ax_tr.legend(); ax_tr.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()


interact(
    plot_sample_gt,
    sample_idx=IntSlider(min=0, max=len(train_ds) - 1, step=1, value=0, description="Sample"),
    time_step=IntSlider(min=0, max=TIME_STEPS - 1, step=1, value=TIME_STEPS // 2, description="Time step"),
    dataset=fixed(train_ds),
)
plt.show()

## §7  Models

We define three groundings, each chosen to match the structural requirements of its corresponding symbol:

- `digit` declares **no structural dimensions**, so it is grounded with a frame-local CNN classifier (`FrameEncoder` plus a linear head). It is applied independently to each frame and has no dependency on neighbouring frames. There is no GRU in this model; the type of `digit` does not support a recurrent grounding.

- `Complete` declares **no structural dimensions**,, so it is also grounded with a per-frame CNN classifier. 

- `appear` consumes `T` and produces no structural dimension (`output_dims=[]`). A frame-local model cannot produce a well-typed whole-video judgement for this predicate. We therefore ground it with a **recurrent classifier**: a per-frame CNN encoder feeds a GRU across time, and the final hidden state is read out into a softmax over `N_CLASSES` (10 digit classes plus `unknown`). The truth value of `appear(x, d)` is computed as the dot product between this softmax vector and the digit vector `d`, as for `=d`.

All three models receive tensors with leading `(batch, T, …)` dimensions.

In [ ]:
class FrameEncoder(nn.Module):
    """Small CNN applied independently to each frame of a (B, T, 1, H, W) video."""

    def __init__(self, out_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 28 -> 14
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 14 -> 7
        )
        self.proj = nn.Sequential(nn.Flatten(), nn.Linear(32 * 7 * 7, out_dim), nn.ReLU())

    def forward(self, x):
        # `digit`/`Complete` declare no free `T` dependence, so the interpreter has
        # already flattened `T` into the batch dim before calling us: x is 4D,
        # (N, 1, H, W). `appear`'s GRU instead needs the *whole* video, so the
        # interpreter passes `x` through untouched: x is 5D, (B, T, 1, H, W). Handle
        # both by flattening/restoring `T` ourselves around the frame-local CNN.
        if x.dim() == 5:
            B, T = x.shape[:2]
            feats = self.proj(self.conv(x.reshape(B * T, *x.shape[2:])))
            return feats.reshape(B, T, -1)
        return self.proj(self.conv(x))


class DigitClassifier(nn.Module):
    """Grounds the `digit` function: frame(s) (B, T, 1, H, W) -> per-frame digit distribution
    (B, T, N_CLASSES). Purely frame-local — no GRU, no dependency across T whatsoever,
    matching `digit`'s signature (no structural dimensions declared at all).

    Outputs N_CLASSES = N_DIGITS + 1 probabilities per frame; the last class (index
    N_DIGITS) is the "unknown" pseudo-class.
    """

    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.head = nn.Linear(feature_dim, n_classes)

    def forward(self, x):
        feats = self.encoder(x)                        # (B, feature_dim)
        return torch.softmax(self.head(feats), dim=-1)  # (B, n_classes)


class CompleteClassifier(nn.Module):
    """Grounds the `Complete` predicate: frame(s) (B, T, 1, H, W) -> truth degree (B, T, 1)."""

    def __init__(self, feature_dim):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.head = nn.Sequential(nn.Linear(feature_dim, 1), nn.Sigmoid())

    def forward(self, x):
        feats = self.encoder(x)          # (..., T, feature_dim)
        return self.head(feats)          # (..., T, 1)


class AppearClassifier(nn.Module):
    """Grounds the whole-video part of `appear`: video (B, T, 1, H, W) -> a single
    softmax distribution (B, N_CLASSES) over "which digit is genuinely appearing, or
    none". A GRU is run across the *entire* time axis and only its final hidden state is
    read out — the model structurally cannot answer without having seen every frame.
    """

    def __init__(self, feature_dim, hidden_dim, n_classes):
        super().__init__()
        self.encoder = FrameEncoder(feature_dim)
        self.rnn = nn.GRU(feature_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        feats = self.encoder(x)            # (B, T, feature_dim)
        _, h_n = self.rnn(feats)           # h_n: (1, B, hidden_dim)
        return torch.softmax(self.head(h_n[-1]), dim=-1)   # (B, n_classes)


digit_model    = DigitClassifier(FRAME_FEATURE_DIM, N_CLASSES).to(device)
complete_model = CompleteClassifier(FRAME_FEATURE_DIM).to(device)
appear_model   = AppearClassifier(FRAME_FEATURE_DIM, RNN_HIDDEN_DIM, N_CLASSES).to(device)

interp["digit"]    = digit_model
interp["Complete"] = complete_model


class AppearPredicate(nn.Module):
    """appear(x, d): dot product between the whole-video softmax and the (typically
    one-hot) digit vector d — the same trick already used for `=d`.

    `x` consumes the whole `T` axis (input_dims=["T"]) while `appear` declares no
    output_dims, so the interpreter passes `x` to `appear_model` with `T` still a real
    leading axis (needed by the GRU), but it reshapes every co-occurring argument —
    including `d`, which has no `T` dependence at all — through that same "preserve T"
    broadcasting path. `d` therefore arrives with a spurious, broadcast-constant `T`
    axis that `appear_model(x)`'s output (which has already consumed and dropped `T`)
    no longer has. Drop that redundant axis before combining.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, d):
        logits = self.model(x)
        d = d.select(1, 0)
        return (logits * d).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)


interp["appear"] = AppearPredicate(appear_model)

# Structural relation `next`: adjacency mask over consecutive time indices.
next_mask = torch.zeros(TIME_STEPS, TIME_STEPS, device=device)
for i in range(TIME_STEPS - 1):
    next_mask[i, i + 1] = 1.0
interp["next"] = next_mask

optimizer = torch.optim.Adam(interp.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
print(interp)

## §8  Training

We train `digit`, `Complete`, and `appear` jointly against the five-clause knowledge base

$$
\mathcal K = \{\delta,\ \phi,\ \forall x:\psi,\ \forall x:\chi,\ \forall x:\beta\}.
$$

At each step, the batch is bound to `x` and `y`, the knowledge base is evaluated, and `kb_backward` aggregates the clause losses using the interpretation's `SatAgg` operator.

Only $\delta$ compares a model prediction directly to ground truth, through `appear(x, y)`. The remaining clauses relate `digit` and `Complete` only to `appear`'s own output:

- $\delta$ pushes `appear(x, y)` towards `1`, providing the only directly supervised signal.
- $\phi$ rewards `digit` for matching the label at every `Complete` frame, provided that `appear` identifies the video as a genuine appearing-digit video. This indirectly pressures `Complete` to fire only once the digit is sufficiently legible.
- $\psi$, $\chi$, and $\beta$ encourage `Complete` to form a monotone, correctly bounded trace and encourage `digit` to abstain on incomplete frames, but only when `is_appear(x)` holds. This gate is itself computed from `appear`.

We report training accuracy as `appear` classification accuracy, namely `argmax_d appear_model(x)[d]` against `argmax(y)`, because this is the only target directly supervised by the loss. Whether `digit` and `Complete` learned their intended frame-level roles is evaluated only after training, in §9–§11, by probing them against `label` and `complete_gt`, which were never used as direct training targets.

In [ ]:
@torch.no_grad()
def classify_appear(interp_):
    """Predict argmax_d appear_model(x)[d] — the only quantity delta directly supervises.
    Index N_DIGITS ("unknown") means "not a genuine appearing-digit video"."""
    return appear_model(interp_["x"]).argmax(dim=-1)   # (B,)


@torch.no_grad()
def classify_digit_at_complete(interp_, complete_gt):
    """Probe `digit`: argmax over real digit classes, at each frame, then majority-vote
    across frames where complete_gt == 1 (falls back to the globally most confident frame
    if a video has no ground-truth-complete frame at all). NEVER used in training."""
    probs = interp_["digit"](interp_["x"])                       # (B, T, N_CLASSES)
    digit_probs = probs[..., :N_DIGITS]                          # (B, T, N_DIGITS)
    per_frame_pred = digit_probs.argmax(dim=-1)                   # (B, T)
    mask = complete_gt.squeeze(-1) > 0.5                          # (B, T)
    preds = []
    for b in range(per_frame_pred.shape[0]):
        idxs = mask[b].nonzero(as_tuple=True)[0]
        if len(idxs) > 0:
            vals, counts = per_frame_pred[b, idxs].unique(return_counts=True)
            preds.append(vals[counts.argmax()].item())
        else:
            preds.append(digit_probs[b].max(dim=-1).values.argmax().item())
    return torch.tensor(preds, device=per_frame_pred.device)


# ── Smoke-test: one forward pass through the full 5-clause KB ─────────────
_batch = next(iter(train_loader))
interp["x"] = _batch["x"].to(device)
interp["y"] = _batch["y"].to(device)

with torch.no_grad():
    sat = interp(phi_kb).value
print(f"KB satisfaction (untrained): {sat.mean().item():.4f}  (expect ~0.5)")
del _batch

In [ ]:
best_val_loss  = float("inf")
patience_count = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(NUM_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    digit_model.train(); complete_model.train(); appear_model.train()
    train_loss = 0.0
    train_preds, train_labels = [], []

    for batch in tqdm(train_loader, desc=f"Ep {epoch+1:3d} [train]", leave=False):
        optimizer.zero_grad()

        interp["x"] = batch["x"].to(device)
        interp["y"] = batch["y"].to(device)

        stats = kb_backward(optimizer, interp, kb_dict, aggregator = "satagg")
        torch.nn.utils.clip_grad_norm_(
            list(digit_model.parameters()) + list(complete_model.parameters())
            + list(appear_model.parameters()),
            MAX_NORM,
        )
        with torch.no_grad():
            preds = classify_appear(interp)
        optimizer.step()

        train_loss += stats["agg_loss"]
        train_preds.extend(preds.cpu().tolist())
        train_labels.extend(batch["y"].argmax(dim=-1).tolist())   # appear ground truth

    train_loss /= len(train_loader)
    train_acc = sum(p == l for p, l in zip(train_preds, train_labels)) / len(train_labels)

    # ── Validate ───────────────────────────────────────────────────────────
    digit_model.eval(); complete_model.eval(); appear_model.eval()
    val_loss = 0.0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            interp["x"] = batch["x"].to(device)
            interp["y"] = batch["y"].to(device)

            sat = interp(phi_kb).value
            val_loss += (1.0 - sat).item()
            val_preds.extend(classify_appear(interp).cpu().tolist())
            val_labels.extend(batch["y"].argmax(dim=-1).tolist())

    val_loss /= len(val_loader)
    val_acc = sum(p == l for p, l in zip(val_preds, val_labels)) / len(val_labels)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    tqdm.write(
        f"Ep {epoch+1:3d} | train loss {train_loss:.4f} "
        f"| train acc {train_acc:.3f} | "
        f"val loss {val_loss:.4f} | val acc {val_acc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss, patience_count = val_loss, 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stop at epoch {epoch+1}")
            break

## §9  Evaluation

We evaluate the directly supervised whole-video predicate `appear`, and then probe the indirectly learned frame-level symbols `digit` and `Complete`. The latter probes use `label` and `complete_gt` only for analysis; these quantities were not used as direct supervision during training.

In [ ]:
# ── Learning curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("Loss (1 - SatAgg)"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["val_acc"], color="tab:green")
axes[1].set_title("Val accuracy"); axes[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# ── (1) `appear` classification report — the ONLY directly-supervised quantity ─────
digit_model.eval(); complete_model.eval(); appear_model.eval()

appear_preds, appear_labels = [], []
digit_preds, digit_labels, digit_is_appear = [], [], []
complete_preds, complete_gts, complete_is_appear = [], [], []

with torch.no_grad():
    for batch in test_loader:
        interp["x"] = batch["x"].to(device)
        complete_gt = batch["complete_gt"].to(device)

        appear_preds.extend(classify_appear(interp).cpu().tolist())
        appear_labels.extend(batch["y"].argmax(dim=-1).tolist())

        digit_preds.extend(classify_digit_at_complete(interp, complete_gt).cpu().tolist())
        digit_labels.extend(batch["label"].tolist())
        digit_is_appear.extend(batch["is_appear"].tolist())

        complete_preds.append(complete_model(interp["x"])[..., 0].cpu())
        complete_gts.append(complete_gt[..., 0].cpu())
        complete_is_appear.extend(batch["is_appear"].tolist())

class_names = [str(d) for d in range(N_DIGITS)] + ["unknown"]
print("=== appear(x, y) classification — directly supervised (delta) ===")
print(classification_report(appear_labels, appear_preds, target_names=class_names, zero_division=0))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay.from_predictions(appear_labels, appear_preds, display_labels=class_names, ax=ax)
ax.set_title("Test confusion matrix — appear(x, y)")
plt.tight_layout(); plt.show()

# ── (2) `digit` probing — NEVER directly supervised, learned only via phi/chi ──────
digit_is_appear = torch.tensor(digit_is_appear).bool()
digit_preds_t   = torch.tensor(digit_preds)
digit_labels_t  = torch.tensor(digit_labels)


def _acc(mask):
    if mask.sum() == 0:
        return float("nan")
    return (digit_preds_t[mask] == digit_labels_t[mask]).float().mean().item()


print("\n=== digit(x[t]) probing accuracy (majority vote at complete_gt==1 frames) ===")
print(f"  overall           : {_acc(torch.ones_like(digit_is_appear)):.3f}")
print(f"  appear videos     : {_acc(digit_is_appear):.3f}")
print(f"  disappear videos  : {_acc(~digit_is_appear):.3f}  (never gated by chi/beta — pure generalisation)")

# ── (3) `Complete` probing — NEVER directly supervised, learned only via psi/beta ──
complete_preds_t = torch.cat(complete_preds, dim=0)          # (N, T)
complete_gts_t   = torch.cat(complete_gts, dim=0)             # (N, T)
complete_is_appear_t = torch.tensor(complete_is_appear).bool()
complete_err = (complete_preds_t - complete_gts_t).abs().mean(dim=-1)   # (N,) mean abs error over t

print("\n=== Complete(x)[t] probing — mean |pred - gt| over t ===")
print(f"  appear videos     : {complete_err[complete_is_appear_t].mean().item():.3f}")
print(f"  disappear videos  : {complete_err[~complete_is_appear_t].mean().item():.3f}  (psi/beta never asserted here)")

## §10  Formula satisfaction at test time

In addition to classification metrics, sLTN allows us to inspect the satisfaction degree of each logical constraint in $\mathcal K$:

- $\delta$ — the directly supervised clause: `appear(x, y)` should be true;
- $\phi$ — label matching at every `Complete` frame, gated by `appear` and `is_appear`;
- $\forall x:\psi(x)$ — temporal persistence of `Complete`, gated by `is_appear`;
- $\forall x:\chi(x)$ — abstention, i.e. prediction of `unknown`, on incomplete frames, gated by `is_appear`;
- $\forall x:\beta(x)$ — boundary condition: incomplete at $t=0$ and complete at $t=T-1$, gated by `is_appear`.

In [ ]:
delta_sats, phi_sats, psi_sats, chi_sats, beta_sats = [], [], [], [], []
with torch.no_grad():
    for batch in tqdm(test_loader):
        interp["x"] = batch["x"].to(device)
        interp["y"] = batch["y"].to(device)

        delta_sats.append(interp(delta).value.item())
        phi_sats.append(interp(phi).value.item())
        psi_sats.append(interp(psi).value.item())
        chi_sats.append(interp(chi).value.item())
        beta_sats.append(interp(beta).value.item())

print(f"Avg delta satisfaction           : {sum(delta_sats)/len(delta_sats):.4f}  (appear(x, y) — directly supervised)")
print(f"Avg phi   satisfaction           : {sum(phi_sats)/len(phi_sats):.4f}  (appear(x,y) & is_appear(x) -> ∀t: Complete -> digit match)")
print(f"Avg (∀x: psi)  satisfaction      : {sum(psi_sats)/len(psi_sats):.4f}  (is_appear(x) -> Complete persistence)")
print(f"Avg (∀x: chi)  satisfaction      : {sum(chi_sats)/len(chi_sats):.4f}  (is_appear(x) -> abstain when incomplete)")
print(f"Avg (∀x: beta) satisfaction      : {sum(beta_sats)/len(beta_sats):.4f}  (is_appear(x) -> boundary condition)")

## §11  Post-training prediction visualisation

For a selected test video, the visualisation shows four panels:

- **Peak frame** — the most revealed frame, together with the true digit label, `digit`'s prediction, the video type (appear or disappear), and `appear`'s whole-video verdict.

- **`digit(x)[t]`** — per-frame probabilities for the 10 real digit classes, plus the `unknown` pseudo-class shown as a dashed gray line. This symbol is never directly supervised; its behaviour is induced through $\phi$ and $\chi$, gated by `appear`.

- **`Complete(x)[t]`** — predicted completeness (green) compared with the ground-truth completeness trace (black dashed). The trace is monotone for appear videos and may rise and fall for disappear videos, where `Complete` is not directly constrained.

- **`appear(x, ·)`** — the directly supervised whole-video softmax over all 11 classes, shown as a bar chart with the true video-level label highlighted.

In [ ]:
@torch.no_grad()
def plot_test_prediction(sample_idx, dataset):
    digit_model.eval(); complete_model.eval(); appear_model.eval()

    sample = dataset[sample_idx]
    x = sample["x"].unsqueeze(0).to(device)          # (1, T, 1, 28, 28)
    time_np = np.arange(TIME_STEPS)
    colors = plt.cm.tab10(np.linspace(0, 1, N_DIGITS))
    kind = "appear" if sample["is_appear"] else "disappear"

    all_probs     = digit_model(x)[0].cpu().numpy()           # (T, N_CLASSES)
    digit_probs   = all_probs[:, :N_DIGITS]                   # (T, N_DIGITS)
    unknown_probs = all_probs[:, N_DIGITS]                    # (T,)
    complete_pred = complete_model(x)[0, :, 0].cpu().numpy()  # (T,)
    complete_gt   = sample["complete_gt"].squeeze(-1).numpy()
    appear_probs  = appear_model(x)[0].cpu().numpy()           # (N_CLASSES,)
    pred_class    = int(digit_probs.max(axis=0).argmax())
    peak_t        = int(complete_gt.argmax()) if complete_gt.max() > 0 else int(np.argmax(digit_probs.max(axis=-1)))

    fig, (ax_img, ax_dig, ax_comp, ax_app) = plt.subplots(1, 4, figsize=(20, 4))

    ax_img.imshow(sample["x"][peak_t, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax_img.set_title(f"peak frame t={peak_t}  (true={sample['label']}, type={kind})\ndigit pred={pred_class}")
    ax_img.axis("off")

    for c in range(N_DIGITS):
        lw = 2.5 if c == sample["label"] else 1
        ax_dig.plot(time_np, digit_probs[:, c], color=colors[c], lw=lw,
                    label=f"{c}" + (" ← true" if c == sample["label"] else ""))
    ax_dig.plot(time_np, unknown_probs, color="gray", lw=1.5, ls="--", label="unknown")
    ax_dig.set_title("digit(x)[t] — never directly supervised")
    ax_dig.set_xlabel("time"); ax_dig.set_ylabel("probability")
    ax_dig.legend(ncols=2, fontsize=7); ax_dig.grid(True, alpha=0.3)

    ax_comp.plot(time_np, complete_gt,   color="black", ls="--", lw=2, label="GT Complete")
    ax_comp.plot(time_np, complete_pred, color="tab:green", lw=2, label="Pred Complete")
    ax_comp.set_title("Complete(x)[t] — never directly supervised"); ax_comp.set_xlabel("time")
    ax_comp.set_ylim(-0.05, 1.1); ax_comp.legend(); ax_comp.grid(True, alpha=0.3)

    bar_colors = ["tab:blue"] * N_DIGITS + ["gray"]
    true_appear_idx = sample["label"] if sample["is_appear"] else N_DIGITS
    bar_colors[true_appear_idx] = "tab:red"
    ax_app.bar(range(N_CLASSES), appear_probs, color=bar_colors)
    ax_app.set_xticks(range(N_CLASSES))
    ax_app.set_xticklabels([str(d) for d in range(N_DIGITS)] + ["unk"])
    ax_app.set_title("appear(x, ·) — directly supervised (δ)")
    ax_app.set_ylim(0, 1.05); ax_app.grid(True, alpha=0.3)

    fig.suptitle(f"Test sample {sample_idx}  (type={kind})")
    plt.tight_layout(); plt.show()


interact(
    plot_test_prediction,
    sample_idx=IntSlider(min=0, max=len(test_ds) - 1, step=1, value=0, description="Test sample"),
    dataset=fixed(test_ds),
)
plt.show()